# Tutorial 2: Standardized Topological Navigation on Non-Convex Glyphs (`'G'`)

This tutorial demonstrates the **Non-Equilibrium Steady State (NESS) Active Matter Regime** of A-Block (Proposition 1, Case 2 & Appendix A / Figure 3).

### Why Non-Convex Letterforms?
The upright sans-serif letter `'G'` contains a sharp inward-pointing horizontal spur and a $180^\circ$ C-hook cavity. Because the conservative restoring force $-\nabla V(\mathbf{x})$ points strictly perpendicular to the stroke boundaries toward the medial axis, pure gradient descent (**Standard Langevin**, $\sigma=0$) arrests inside the initial spur pocket, leaving $>66\%$ of the glyph unvisited.

When coupled with the Clifford bivector rotor $\sigma I \nabla V(\mathbf{x})$, particles circulate tangentially along the stroke boundaries while the dilator $-\kappa \nabla V(\mathbf{x})$ clamps them inside the glyph contour.

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from reproduce_typography_figure3 import build_glyph_potential, simulate_typography_ensemble
from src.metrics import calc_glyph_metrics

In [ ]:
mask, V, grad_x, grad_y, r_dish = build_glyph_potential(size=400, char='G')
print(f'Constructed 400x400 glyph potential | Circular aperture radius r_dish = {r_dish} px')

configs = [
    ('Brownian Dynamics (f = 0)',        0.0,  0.0, 'brownian'),
    ('Standard Langevin (f = -grad V)',  0.0,  1.0, 'langevin'),
    ('Rotor-Only (kappa = 0)',          11.0,  0.0, 'rotor'),
    ('A-Block Full Clifford (Ours)',    11.0,  1.8, 'ablock'),
]

print(f"{'Method':<34} | {'Precision (%)':<15} | {'Traj Visitation (%)':<20} | {'Terminal Cov (%)'}")
print('-' * 92)
for name, sigma, kappa, mode in configs:
    pts, traj, visited_grid = simulate_typography_ensemble(
        grad_x, grad_y, mask, r_dish=r_dish, N=1200, T=4000, dt=2.0,
        sigma=sigma, kappa=kappa, noise_amp=0.85, seed=42, mode=mode
    )
    prec, term_cov, visit_cov = calc_glyph_metrics(pts, mask, visited_grid=visited_grid, radius=8)
    print(f"{name:<34} | {prec:<14.1f}% | {visit_cov:<19.1f}% | {term_cov:.1f}%")